<a href="https://colab.research.google.com/github/MauricioLlugdar/CoffeeShop-Sales/blob/main/01_CoffeeShopSales_Analysis_Exploration_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Coffee Shop Sales (Dirty Dataset) - Analysis, Exploration and Cleaning


## Work plan:

**Initial stage:** analyze the data carefully, perform cleaning and feature engineering.

**Action:** after the initial stage, the plan is to predict the quantity sold during the following week. To do this, we will need to create the necessary features for prediction.

**Extra:** ideally, we would analyze how many units of each item will be sold, so we can estimate the required stock. To achieve this, other design decisions need to be made.


In [64]:
!pip install kagglehub

In [65]:
import kagglehub
import pandas as pd
import numpy as np
from pathlib import Path


In [66]:
path = kagglehub.dataset_download("ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'cafe-sales-dirty-data-for-cleaning-training' dataset.
Path to dataset files: /kaggle/input/cafe-sales-dirty-data-for-cleaning-training


In [67]:
df_raw = pd.read_csv(path + "/dirty_cafe_sales.csv")
df = df_raw.copy()
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


### General dataset analysis


In [68]:
df.shape

(10000, 8)

In [69]:
df.columns

Index(['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent',
       'Payment Method', 'Location', 'Transaction Date'],
      dtype='object')

In [70]:
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_9226047,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [71]:
df.isna().mean().sort_values(ascending=False)

,0
Location,0.3265
Payment Method,0.2579
Item,0.0333
Price Per Unit,0.0179
Total Spent,0.0173
Transaction Date,0.0159
Quantity,0.0138
Transaction ID,0.0000


In [72]:
print(df.duplicated().sum())


0


### Analysis of each feature


In [73]:
print(df.nunique().sort_values())

Location                4
Payment Method          5
Quantity                7
Price Per Unit          8
Item                   10
Total Spent            19
Transaction Date      367
Transaction ID      10000
dtype: int64


In [74]:
exclude_cols = ['Transaction ID', 'Transaction Date']

for col in df.columns:
    if col not in exclude_cols:
        unique_vals = df[col].unique()
        print(f"{col} ({len(unique_vals)} unique values):\n {unique_vals}\n")
    else:
        unique_vals = df[col].unique()[:5]
        print(f"{col} (Showing 5 out of {len(df[col].unique())}):\n {unique_vals}...\n")


Transaction ID (Showing 5 out of 10000):
 ['TXN_1961373' 'TXN_4977031' 'TXN_4271903' 'TXN_7034554' 'TXN_3160411']...

Item (11 unique values):
 ['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'UNKNOWN' 'Sandwich' nan
 'ERROR' 'Juice' 'Tea']

Quantity (8 unique values):
 ['2' '4' '5' '3' '1' 'ERROR' 'UNKNOWN' nan]

Price Per Unit (9 unique values):
 ['2.0' '3.0' '1.0' '5.0' '4.0' '1.5' nan 'ERROR' 'UNKNOWN']

Total Spent (20 unique values):
 ['4.0' '12.0' 'ERROR' '10.0' '20.0' '9.0' '16.0' '15.0' '25.0' '8.0' '5.0'
 '3.0' '6.0' nan 'UNKNOWN' '2.0' '1.0' '7.5' '4.5' '1.5']

Payment Method (6 unique values):
 ['Credit Card' 'Cash' 'UNKNOWN' 'Digital Wallet' 'ERROR' nan]

Location (5 unique values):
 ['Takeaway' 'In-store' 'UNKNOWN' nan 'ERROR']

Transaction Date (Showing 5 out of 368):
 ['2023-09-08' '2023-05-16' '2023-07-19' '2023-04-27' '2023-06-11']...



In [75]:
df["Transaction ID"].nunique()


10000

In [76]:
df["Transaction ID"].duplicated().sum()

np.int64(0)

Since there are no duplicated transactions or loading errors, I remove this feature because it will not be useful for prediction.


In [77]:
df = df.drop(columns=["Transaction ID"])

The best approach for categorical variables is to apply One-Hot Encoding so the model can use them for prediction. The categorical variables available in the dataset are given by the following features:

*   Item
*   Payment Method
*   Location


First, I unify errors and unknown values as NaN. I also clean string and numeric values.


In [78]:
df = df.replace(["UNKNOWN", "ERROR"], np.nan)

for col in ["Item", "Payment Method", "Location"]:
    df[col] = df[col].astype("string").str.strip()

for col in ["Quantity", "Price Per Unit", "Total Spent"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")


Let's see the result of this cleaning step.


In [79]:
for col in df.columns:
    if col not in exclude_cols:
        unique_vals = df[col].unique()
        print(f"{col} ({len(unique_vals)} unique values):\n {unique_vals}\n")
    else:
        unique_vals = df[col].unique()[:5]
        print(f"{col} (Showing 5 out of {len(df[col].unique())}):\n {unique_vals}...\n")


Item (9 unique values):
 <StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',       <NA>,
 'Sandwich',    'Juice',      'Tea']
Length: 9, dtype: string

Quantity (6 unique values):
 [ 2.  4.  5.  3.  1. nan]

Price Per Unit (7 unique values):
 [2.  3.  1.  5.  4.  1.5 nan]

Total Spent (18 unique values):
 [ 4.  12.   nan 10.  20.   9.  16.  15.  25.   8.   5.   3.   6.   2.
  1.   7.5  4.5  1.5]

Payment Method (4 unique values):
 <StringArray>
['Credit Card', 'Cash', <NA>, 'Digital Wallet']
Length: 4, dtype: string

Location (3 unique values):
 <StringArray>
['Takeaway', 'In-store', <NA>]
Length: 3, dtype: string

Transaction Date (Showing 5 out of 366):
 <DatetimeArray>
['2023-09-08 00:00:00', '2023-05-16 00:00:00', '2023-07-19 00:00:00',
 '2023-04-27 00:00:00', '2023-06-11 00:00:00']
Length: 5, dtype: datetime64[ns]...



Let's see the head of the dataframe so far.


In [80]:
df.head()

,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19
3,Salad,2.0,5.0,10.0,<NA>,<NA>,2023-04-27
4,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


### Analysis of Numerical Variables

As can be seen, the NaN value of Total Spent in row 2 can easily be replaced by Quantity * Price Per Unit to improve data completeness.


In [81]:
df.loc[df["Total Spent"] != df["Quantity"] * df["Price Per Unit"]]

,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
2,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19
20,Smoothie,NaN,4.0,20.0,Cash,In-store,2023-04-04
25,Smoothie,3.0,4.0,NaN,<NA>,<NA>,2023-12-13
31,<NA>,2.0,1.0,NaN,Credit Card,<NA>,2023-11-06
42,Tea,2.0,1.5,NaN,<NA>,Takeaway,2023-01-10
...,...,...,...,...,...,...,...
9984,Smoothie,NaN,4.0,4.0,Cash,Takeaway,2023-07-27
9988,Cake,5.0,3.0,NaN,<NA>,<NA>,NaT
9993,Smoothie,2.0,4.0,NaN,Cash,<NA>,2023-10-20
9996,<NA>,3.0,NaN,3.0,Digital Wallet,<NA>,2023-06-02


As we can see, we need to fill values where Total Spent is NaN and we have quantity and price. We also need to fill variables whose value can be inferred from the rest of the row.


In [82]:
# 1. We create a mask to identify where we can calculate the total.
# We need Quantity and Price Per Unit to be non-null.
mask_can_calculate = df['Quantity'].notna() & df['Price Per Unit'].notna()

# 2. Within those rows, we look for the ones that are wrong:
# the value is different from the product or it is NaN.
mask_error = (df['Total Spent'] != (df['Quantity'] * df['Price Per Unit']))

# We combine both conditions and update only when we have the data and there is an error/null value.
final_condition = mask_can_calculate & mask_error

df.loc[final_condition, 'Total Spent'] = df['Quantity'] * df['Price Per Unit']

print(f"Corrected {final_condition.sum()} rows.")


Corrected 462 rows.


In [83]:
df.head()

,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,Salad,2.0,5.0,10.0,<NA>,<NA>,2023-04-27
4,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


There may be cases where Quantity or Price Per Unit can be inferred, so we will try to do that.


Case where Quantity is NaN and can be inferred.


In [84]:
mask_quantity = df['Quantity'].isna() & df["Total Spent"].notna() & df['Price Per Unit'].notna()

print(df.loc[mask_quantity].head())

         Item  Quantity  Price Per Unit  Total Spent  Payment Method  \
20   Smoothie       NaN             4.0         20.0            Cash   
55     Cookie       NaN             1.0          2.0     Credit Card   
57       Cake       NaN             3.0          3.0  Digital Wallet   
66      Juice       NaN             3.0          6.0            Cash   
117     Juice       NaN             3.0          9.0  Digital Wallet   

     Location Transaction Date  
20   In-store       2023-04-04  
55   Takeaway       2023-03-19  
57   In-store       2023-04-19  
66       <NA>       2023-03-30  
117      <NA>       2023-01-10  


As we can see, the value of Quantity can be inferred.


In [85]:
df.loc[mask_quantity, "Quantity"] = df['Total Spent'] * (1/df['Price Per Unit'])

mask_quantity = df['Quantity'].isna() & df["Total Spent"].notna() & df['Price Per Unit'].notna()
print(mask_quantity.sum())

0


As we can see, there are no Quantity values left that can be corrected.

Now let's correct Price Per Unit.


In [86]:
mask_ppp = df['Price Per Unit'].isna() & df["Total Spent"].notna() & df['Quantity'].notna()

print(df.loc[mask_ppp].head())

      Item  Quantity  Price Per Unit  Total Spent Payment Method  Location  \
56    Cake       5.0             NaN         15.0           <NA>  Takeaway   
68   Salad       2.0             NaN         10.0           <NA>  In-store   
85     Tea       3.0             NaN          4.5           Cash      <NA>   
104  Juice       2.0             NaN          6.0           <NA>      <NA>   
118   <NA>       5.0             NaN         15.0           <NA>  In-store   

    Transaction Date  
56        2023-06-27  
68        2023-10-27  
85        2023-10-29  
104              NaT  
118       2023-02-06  


In [87]:
df.loc[mask_ppp, "Price Per Unit"] = df['Total Spent'] * (1/df['Quantity'])

mask_ppp = df['Price Per Unit'].isna() & df["Total Spent"].notna() & df['Quantity'].notna()
print(mask_ppp.sum())

0


Let's now check the missing data, to compare it with the beginning of the dataset analysis.


In [88]:
df.isna().mean().sort_values(ascending=False)

,0
Location,0.3961
Payment Method,0.3178
Item,0.0969
Transaction Date,0.0460
Total Spent,0.0040
Price Per Unit,0.0038
Quantity,0.0038


As we can see, there is an important decrease in rows where Quantity, Price Per Unit and Total Spent values are missing.


### Analysis of Categorical Variables


As we can see, 39.61% of Location values and 31.78% of Payment Method values are missing. This may be telling us something, so we will not fill them with the mode, nor will we discard the rows where they are missing. We will use another category to refer to these cases.

For the Item variable, given its quantity and unit price, it may be possible to infer what the item is. This would allow us to recover several Items and reduce the number of NaN values.

Finally, the Transaction Date feature is essential for our time-based prediction, because without it we cannot predict consumption for the following week.


In [89]:
df = df.dropna(subset=['Transaction Date'])
df.loc[df["Location"].isna(), "Location"] = "Unknown Location"
df.loc[df["Payment Method"].isna(), "Payment Method"] = "Unknown Payment Method"

In [90]:
df.isna().mean().sort_values(ascending=False)

,0
Item,0.097170
Total Spent,0.004088
Quantity,0.003774
Price Per Unit,0.003669
Payment Method,0.000000
Location,0.000000
Transaction Date,0.000000


#### Design Decision
This is where two design decisions are separated, because we will want to predict how many items will be sold the following week based on past data. Separately, we will also want to predict how much of each item will be sold, so if some items are missing in some rows, it will not be possible to predict properly.

Before this, let's first analyze if there are rows with unrecoverable quantities and remove them, because they will not be useful for predicting anything.


In [91]:
df["Quantity"].isna().sum()

np.int64(36)

In [92]:
df.loc[df["Quantity"].isna()].head()

,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
236,Salad,NaN,5.0,NaN,Unknown Payment Method,In-store,2023-05-18
278,Juice,NaN,3.0,NaN,Cash,Takeaway,2023-04-15
629,Cake,NaN,NaN,12.0,Digital Wallet,In-store,2023-12-30
641,Juice,NaN,3.0,NaN,Unknown Payment Method,Unknown Location,2023-03-17
738,Sandwich,NaN,4.0,NaN,Unknown Payment Method,Takeaway,2023-05-14


In [93]:
df = df.dropna(subset=['Quantity'])
df.isna().mean().sort_values(ascending=False)

,0
Item,0.097222
Price Per Unit,0.001999
Total Spent,0.001999
Quantity,0.000000
Payment Method,0.000000
Location,0.000000
Transaction Date,0.000000


As we can see, the number of missing values in Price Per Unit and Total Spent is the same. We can try to obtain the missing values if their rows indicate the item and quantity, since the values of each item are fixed in the dataset description.


Prices

*   Coffee    $2

*   Tea       $1.5

*   Sandwich  $4

*   Salad     $5

*   Cake      $3

*   Cookie    $1

*   Smoothie  $4

*   Juice     $3


In [94]:
mask_item_for_unit_price = df['Price Per Unit'].isna() & df["Item"].notna() # we already know Quantity is complete

price_map = {
    "Coffee": 2.0,
    "Tea": 1.5,
    "Sandwich": 4.0,
    "Salad": 5.0,
    "Cake": 3.0,
    "Cookie": 1.0,
    "Smoothie": 4.0,
    "Juice": 3.0
}

df.loc[mask_item_for_unit_price, "Price Per Unit"] = (
    df.loc[mask_item_for_unit_price, "Item"].map(price_map)
)


In [95]:
mask_error = ((df['Total Spent'] != (df['Quantity'] * df['Price Per Unit'])) & df["Price Per Unit"].notna())
df.loc[mask_error, "Total Spent"] = df['Quantity'] * df['Price Per Unit']

df.isna().mean().sort_values(ascending=False)

,0
Item,0.097222
Price Per Unit,0.000316
Total Spent,0.000316
Quantity,0.000000
Payment Method,0.000000
Location,0.000000
Transaction Date,0.000000


As we can see, the number of NaN values in Total Spent and Price Per Unit is the same. Let's inspect these cases and see if anything can be done to recover them.


In [96]:
df.loc[df["Price Per Unit"].isna()].head()

,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
1761,<NA>,4.0,NaN,NaN,Credit Card,Unknown Location,2023-02-09
2289,<NA>,4.0,NaN,NaN,Unknown Payment Method,Unknown Location,2023-12-09
4152,<NA>,2.0,NaN,NaN,Unknown Payment Method,In-store,2023-12-14


In [97]:
df["Price Per Unit"].isna().sum()

np.int64(3)

In [98]:
df.loc[df["Total Spent"].isna()].head()

,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
1761,<NA>,4.0,NaN,NaN,Credit Card,Unknown Location,2023-02-09
2289,<NA>,4.0,NaN,NaN,Unknown Payment Method,Unknown Location,2023-12-09
4152,<NA>,2.0,NaN,NaN,Unknown Payment Method,In-store,2023-12-14


In [99]:
df["Total Spent"].isna().sum()

np.int64(3)

As we can see, there are 3 rows, but their values cannot be obtained. However, they can still be useful when analyzing the total quantity sold the following week, but not when analyzing the quantity sold for each product the following week. Therefore, for item-level analysis we discard them, and for total analysis we keep them.


In [100]:
df_total_quantity = df.copy()
df_item_quantity = df.dropna(subset="Price Per Unit").copy()


In [101]:
df_item_quantity.isna().mean().sort_values(ascending=False)


,0
Item,0.096937
Quantity,0.000000
Price Per Unit,0.000000
Total Spent,0.000000
Payment Method,0.000000
Location,0.000000
Transaction Date,0.000000


In [102]:
df_total_quantity.isna().mean().sort_values(ascending=False)


,0
Item,0.097222
Price Per Unit,0.000316
Total Spent,0.000316
Quantity,0.000000
Payment Method,0.000000
Location,0.000000
Transaction Date,0.000000


Finally, for the item-level DataFrame, we still need to obtain the missing Items that can be inferred, either directly or probabilistically using the current proportions. If we removed those rows, we would lose 10% of our dataset, which would be too radical.

**Clarification:** for the DataFrame where we want to calculate the total quantity of items that will be sold, knowing which item it was does not affect us.


In [103]:
deterministic_price_to_item = {
    1.0: "Cookie",
    1.5: "Tea",
    2.0: "Coffee",
    5.0: "Salad"
}
ambiguous_price_to_items = {
    3.0: ["Juice", "Cake"],
    4.0: ["Sandwich", "Smoothie"]
}

mask_unit_price_to_item = df_item_quantity['Item'].isna() & df_item_quantity["Price Per Unit"].notna()


In [104]:
df_item_quantity.loc[df_item_quantity['Price Per Unit'].isin(deterministic_price_to_item.keys()) & mask_unit_price_to_item].head()


,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
14,<NA>,2.0,1.5,3.0,Unknown Payment Method,In-store,2023-03-21
30,<NA>,5.0,2.0,10.0,Digital Wallet,Unknown Location,2023-06-02
31,<NA>,2.0,1.0,2.0,Credit Card,Unknown Location,2023-11-06
52,<NA>,5.0,5.0,25.0,Digital Wallet,Unknown Location,2023-03-15
63,<NA>,3.0,5.0,15.0,Unknown Payment Method,Takeaway,2023-11-18


In [105]:
df_item_quantity.loc[df_item_quantity['Price Per Unit'].isin(deterministic_price_to_item.keys()) & mask_unit_price_to_item, "Item"] = df_item_quantity.loc[df_item_quantity['Price Per Unit'].isin(deterministic_price_to_item.keys()) & mask_unit_price_to_item, "Price Per Unit"].map(deterministic_price_to_item)
df_item_quantity.loc[df_item_quantity['Price Per Unit'].isin(deterministic_price_to_item.keys()) & mask_unit_price_to_item].head()


,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
14,Tea,2.0,1.5,3.0,Unknown Payment Method,In-store,2023-03-21
30,Coffee,5.0,2.0,10.0,Digital Wallet,Unknown Location,2023-06-02
31,Cookie,2.0,1.0,2.0,Credit Card,Unknown Location,2023-11-06
52,Salad,5.0,5.0,25.0,Digital Wallet,Unknown Location,2023-03-15
63,Salad,3.0,5.0,15.0,Unknown Payment Method,Takeaway,2023-11-18


As we can see, we completed the most direct step: assigning the Item value to rows where it can be directly inferred from Price Per Unit.

Now let's assign the remaining Item values probabilistically.


In [106]:
df_item_quantity.isna().mean().sort_values(ascending=False)


,0
Item,0.047363
Quantity,0.000000
Price Per Unit,0.000000
Total Spent,0.000000
Payment Method,0.000000
Location,0.000000
Transaction Date,0.000000


We analyze the proportions of the Items with the same price.


In [107]:
# PPP = 3 & Item = Cake
cake_count = len(df_item_quantity.loc[(df_item_quantity["Price Per Unit"] == 3) &
 (df_item_quantity['Item'] == "Cake")])

# PPP = 3 & Item = Juice
juice_count = len(df_item_quantity.loc[(df_item_quantity["Price Per Unit"] == 3) &
 (df_item_quantity['Item'] == "Juice")])

# PPP = 3 & Item != NaN
all_non_missing_three = len(df_item_quantity.loc[(df_item_quantity["Price Per Unit"] == 3) &
 (df_item_quantity['Item'].notna())])

# PPP = 3 & Item == NaN
df_3_nan = df_item_quantity.loc[(df_item_quantity["Price Per Unit"] == 3) &
 (df_item_quantity['Item'].isna())]


# Proportions
cake_prop = cake_count / all_non_missing_three
juice_prop = juice_count / all_non_missing_three

print(f"Cake proportion: {cake_prop}")
print(f"Juice proportion: {juice_prop}")

print("")
print("Total NaN count:")
print((df_item_quantity['Item'].isna()).sum())
print("")
print("Counts for both groups:")
print(((df_item_quantity["Price Per Unit"] == 3) &
 (df_item_quantity['Item'].isna())).sum())
print(((df_item_quantity["Price Per Unit"] == 4) &
 (df_item_quantity['Item'].isna())).sum())


Cake proportion: 0.49022282855843563
Juice proportion: 0.5097771714415643

Total NaN count:
450

Counts for both groups:
234
216


In [108]:
# PPP = 4 & Item = Sandwich
sandwich_count = len(df_item_quantity.loc[(df_item_quantity["Price Per Unit"] == 4) &
 (df_item_quantity['Item'] == "Sandwich")])

# PPP = 4 & Item = Smoothie
smoothie_count = len(df_item_quantity.loc[(df_item_quantity["Price Per Unit"] == 4) &
 (df_item_quantity['Item'] == "Smoothie")])

# PPP = 4 & Item != NaN
all_non_missing_four = len(df_item_quantity.loc[(df_item_quantity["Price Per Unit"] == 4) &
 (df_item_quantity['Item'].notna())])

# Proportions
sandwich_prop = sandwich_count / all_non_missing_four
smoothie_prop = smoothie_count / all_non_missing_four

print(f"Sandwich proportion: {sandwich_prop}")
print(f"Smoothie proportion: {smoothie_prop}")


Sandwich proportion: 0.5061494796594135
Smoothie proportion: 0.4938505203405866


We already have the proportions, so we will use them to assign values to the Items that are NaN. We will use a random generator to decide which value each item will take depending on its Price Per Unit and proportion.


In [109]:
rng = np.random.default_rng(8)

mask_3_missing = (
    (df_item_quantity["Price Per Unit"] == 3)
    & (df_item_quantity["Item"].isna())
)

n_3 = mask_3_missing.sum()

df_item_quantity.loc[mask_3_missing, "Item"] = rng.choice(
    ["Cake", "Juice"],
    size=n_3,
    p=[cake_prop, juice_prop]
)

mask_4_missing = (
    (df_item_quantity["Price Per Unit"] == 4)
    & (df_item_quantity["Item"].isna())
)

n_4 = mask_4_missing.sum()

df_item_quantity.loc[mask_4_missing, "Item"] = rng.choice(
    ["Sandwich", "Smoothie"],
    size=n_4,
    p=[sandwich_prop, smoothie_prop]
)


Let's see if there are missing values left or if the data is already ready to use in a model.


In [110]:
df_item_quantity.isna().mean().sort_values(ascending=False)


,0
Item,0.0
Quantity,0.0
Price Per Unit,0.0
Total Spent,0.0
Payment Method,0.0
Location,0.0
Transaction Date,0.0


The next step is to create the column that will be our target, "Sales next week". In the case where we want to analyze total sales, we do not care which item was sold, but how many units were sold, so we will reduce our DataFrame.


Now let's check the DataFrame of "df_item_quantity" where we want to be able to differentiate the quantity of items sold in the next week

In [111]:
df_item_quantity["week"] = (
    df_item_quantity["Transaction Date"]
    .dt.to_period("W")
    .apply(lambda r: r.start_time)
)

df_item_quantity.head()

,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,week
0,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08,2023-09-04
1,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16,2023-05-15
2,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19,2023-07-17
3,Salad,2.0,5.0,10.0,Unknown Payment Method,Unknown Location,2023-04-27,2023-04-24
4,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11,2023-06-05


We want to group by week not by transaction, let´s create a new DataFrame to show this

In [112]:
df_weekly_item_quantity = (
    df_item_quantity
    .groupby(["week", "Item"])
    .agg(
        units_sold=("Quantity", "sum"),
        n_transactions=("Quantity", "count")
    )
    .reset_index()
)

df_weekly_item_quantity.head()

,week,Item,units_sold,n_transactions
0,2022-12-26,Cake,14.0,4
1,2022-12-26,Coffee,14.0,5
2,2022-12-26,Cookie,4.0,1
3,2022-12-26,Juice,6.0,2
4,2022-12-26,Salad,4.0,2


Finally I have to add the target columns for each Item

In [113]:
# For representing the seasonality
target_week = df_weekly_item_quantity["week"] + pd.Timedelta(weeks=1)

target_day_of_year = target_week.dt.dayofyear

WEEKS_PER_YEAR = 52.1775

df_weekly_item_quantity["target_year_sin"] = np.sin(
    2 * np.pi * target_day_of_year / WEEKS_PER_YEAR
)

df_weekly_item_quantity["target_year_cos"] = np.cos(
    2 * np.pi * target_day_of_year / WEEKS_PER_YEAR
)

df_weekly_item_quantity.head()

,week,Item,units_sold,n_transactions,target_year_sin,target_year_cos
0,2022-12-26,Cake,14.0,4,0.238517,0.971138
1,2022-12-26,Coffee,14.0,5,0.238517,0.971138
2,2022-12-26,Cookie,4.0,1,0.238517,0.971138
3,2022-12-26,Juice,6.0,2,0.238517,0.971138
4,2022-12-26,Salad,4.0,2,0.238517,0.971138


We have to represent the Items that didn't were sold each week

In [114]:
all_weeks = pd.date_range(
    start=df_weekly_item_quantity["week"].min(),
    end=df_weekly_item_quantity["week"].max(),
    freq="7D"
)

all_items = df_weekly_item_quantity["Item"].unique()

full_index = pd.MultiIndex.from_product(
    [all_weeks, all_items],
    names=["week", "Item"]
)

df_weekly_item_quantity = (
    df_weekly_item_quantity
    .set_index(["week", "Item"])
    .reindex(full_index)
    .reset_index()
)

df_weekly_item_quantity["units_sold"] = (
    df_weekly_item_quantity["units_sold"].fillna(0)
)

df_weekly_item_quantity.head(10)

,week,Item,units_sold,n_transactions,target_year_sin,target_year_cos
0,2022-12-26,Cake,14.0,4,0.238517,0.971138
1,2022-12-26,Coffee,14.0,5,0.238517,0.971138
2,2022-12-26,Cookie,4.0,1,0.238517,0.971138
3,2022-12-26,Juice,6.0,2,0.238517,0.971138
4,2022-12-26,Salad,4.0,2,0.238517,0.971138
5,2022-12-26,Sandwich,11.0,3,0.238517,0.971138
6,2022-12-26,Smoothie,2.0,1,0.238517,0.971138
7,2022-12-26,Tea,8.0,2,0.238517,0.971138
8,2023-01-02,Cake,67.0,22,0.883731,0.467996
9,2023-01-02,Coffee,66.0,20,0.883731,0.467996


In [115]:
df_weekly_item_quantity['target_next_week_units'] = (
    df_weekly_item_quantity.groupby('Item')['units_sold']
    .shift(-1)
)

for window in [4, 8]:
    df_weekly_item_quantity[f'avg_last_{window}_weeks'] = (
        df_weekly_item_quantity.groupby('Item')['units_sold']
        .transform(lambda x: x.rolling(window=window, min_periods=1).mean())
    )

display(df_weekly_item_quantity.head(10))

,week,Item,units_sold,n_transactions,target_year_sin,target_year_cos,target_next_week_units,avg_last_4_weeks,avg_last_8_weeks
0,2022-12-26,Cake,14.0,4,0.238517,0.971138,67.0,14.0,14.0
1,2022-12-26,Coffee,14.0,5,0.238517,0.971138,66.0,14.0,14.0
2,2022-12-26,Cookie,4.0,1,0.238517,0.971138,59.0,4.0,4.0
3,2022-12-26,Juice,6.0,2,0.238517,0.971138,64.0,6.0,6.0
4,2022-12-26,Salad,4.0,2,0.238517,0.971138,83.0,4.0,4.0
5,2022-12-26,Sandwich,11.0,3,0.238517,0.971138,85.0,11.0,11.0
6,2022-12-26,Smoothie,2.0,1,0.238517,0.971138,63.0,2.0,2.0
7,2022-12-26,Tea,8.0,2,0.238517,0.971138,71.0,8.0,8.0
8,2023-01-02,Cake,67.0,22,0.883731,0.467996,51.0,40.5,40.5
9,2023-01-02,Coffee,66.0,20,0.883731,0.467996,73.0,40.0,40.0


Let's see how many Items have NaN as the target_next_week_units

In [116]:
df_weekly_item_quantity["target_next_week_units"].isna().sum()

np.int64(8)

In [117]:
df_last_week_item_quantity = df_weekly_item_quantity.loc[df_weekly_item_quantity["target_next_week_units"].isna()].copy()
df_last_week_item_quantity.head(8)

,week,Item,units_sold,n_transactions,target_year_sin,target_year_cos,target_next_week_units,avg_last_4_weeks,avg_last_8_weeks
416,2023-12-25,Cake,64.0,22,0.120129,0.992758,NaN,60.50,73.250
417,2023-12-25,Coffee,49.0,17,0.120129,0.992758,NaN,71.25,72.625
418,2023-12-25,Cookie,66.0,19,0.120129,0.992758,NaN,64.00,66.500
419,2023-12-25,Juice,68.0,24,0.120129,0.992758,NaN,74.25,71.750
420,2023-12-25,Salad,57.0,19,0.120129,0.992758,NaN,60.25,69.625
421,2023-12-25,Sandwich,60.0,20,0.120129,0.992758,NaN,72.00,67.500
422,2023-12-25,Smoothie,71.0,22,0.120129,0.992758,NaN,69.00,73.250
423,2023-12-25,Tea,80.0,26,0.120129,0.992758,NaN,61.75,62.000


As we can see the Items that have the column target_next_week_units with the value NaN are from the last week of the dataset for obvious reasons. We have to drop them to train our model and test it. Nevertheless we can try to predict with our model the values for each Item.

In [118]:
df_weekly_item_quantity = df_weekly_item_quantity.dropna(subset=["target_next_week_units"])
df_weekly_item_quantity.isna().mean().sort_values(ascending=False)

,0
week,0.0
Item,0.0
units_sold,0.0
n_transactions,0.0
target_year_sin,0.0
target_year_cos,0.0
target_next_week_units,0.0
avg_last_4_weeks,0.0
avg_last_8_weeks,0.0


Finally, we managed to completely clean the dataset, obtaining 2 DataFrames: one to calculate the total quantity sold the following week and another to calculate the quantity sold for each item the following week.

Now let's apply One-Hot Encoding to the categorical variables of the second DataFrame mentioned before.


In [119]:
df_weekly_item_quantity_encoded = pd.get_dummies(
    df_weekly_item_quantity,
    columns=["Item"],
    drop_first=True
)

df_weekly_item_quantity_encoded.head(10)

,week,units_sold,n_transactions,target_year_sin,target_year_cos,target_next_week_units,avg_last_4_weeks,avg_last_8_weeks,Item_Coffee,Item_Cookie,Item_Juice,Item_Salad,Item_Sandwich,Item_Smoothie,Item_Tea
0,2022-12-26,14.0,4,0.238517,0.971138,67.0,14.0,14.0,False,False,False,False,False,False,False
1,2022-12-26,14.0,5,0.238517,0.971138,66.0,14.0,14.0,True,False,False,False,False,False,False
2,2022-12-26,4.0,1,0.238517,0.971138,59.0,4.0,4.0,False,True,False,False,False,False,False
3,2022-12-26,6.0,2,0.238517,0.971138,64.0,6.0,6.0,False,False,True,False,False,False,False
4,2022-12-26,4.0,2,0.238517,0.971138,83.0,4.0,4.0,False,False,False,True,False,False,False
5,2022-12-26,11.0,3,0.238517,0.971138,85.0,11.0,11.0,False,False,False,False,True,False,False
6,2022-12-26,2.0,1,0.238517,0.971138,63.0,2.0,2.0,False,False,False,False,False,True,False
7,2022-12-26,8.0,2,0.238517,0.971138,71.0,8.0,8.0,False,False,False,False,False,False,True
8,2023-01-02,67.0,22,0.883731,0.467996,51.0,40.5,40.5,False,False,False,False,False,False,False
9,2023-01-02,66.0,20,0.883731,0.467996,73.0,40.0,40.0,True,False,False,False,False,False,False


Do the same for the total quantity Data Frame

In [120]:
df_total_quantity["week"] = (
    df_total_quantity["Transaction Date"]
    .dt.to_period("W")
    .apply(lambda r: r.start_time)
)

df_weekly_total = (
    df_total_quantity
    .groupby("week")
    .agg(
        units_sold=("Quantity", "sum"),
        n_transactions=("Quantity", "count")
    )
    .reset_index()
)

df_weekly_total = df_weekly_total.sort_values("week")

all_weeks = pd.date_range(
    start=df_weekly_total["week"].min(),
    end=df_weekly_total["week"].max(),
    freq="W-MON"
)

# Fill with 0 the weeks that doesn't appear in the dataset
df_weekly_total = (
    df_weekly_total
    .set_index("week")
    .reindex(all_weeks, fill_value=0)
    .rename_axis("week")
    .reset_index()
)

df_weekly_total["target_next_week_units"] = (
    df_weekly_total["units_sold"].shift(-1)
)

df_weekly_total["avg_last_4_weeks"] = (
    df_weekly_total["units_sold"]
    .rolling(window=4, min_periods=1)
    .mean()
)

df_weekly_total["avg_last_8_weeks"] = (
    df_weekly_total["units_sold"]
    .rolling(window=8, min_periods=1)
    .mean()
)

df_weekly_total["historical_avg_units"] = (
    df_weekly_total["units_sold"]
    .expanding()
    .mean()
)

target_week = df_weekly_total["week"] + pd.Timedelta(weeks=1)

target_day_of_year = target_week.dt.dayofyear

# For representing the seasonality
df_weekly_total["target_year_sin"] = np.sin(
    2 * np.pi * target_day_of_year / WEEKS_PER_YEAR
)

df_weekly_total["target_year_cos"] = np.cos(
    2 * np.pi * target_day_of_year / WEEKS_PER_YEAR
)

# I reorder the columns to leave the target as the last one.
cols = [col for col in df_weekly_total.columns if col != "target_next_week_units"]
cols = cols + ["target_next_week_units"]

df_weekly_total = df_weekly_total[cols]

display(df_weekly_total.head())

,week,units_sold,n_transactions,avg_last_4_weeks,avg_last_8_weeks,historical_avg_units,target_year_sin,target_year_cos,target_next_week_units
0,2022-12-26,63.0,20,63.00,63.00,63.00,0.238517,0.971138,558.0
1,2023-01-02,558.0,186,310.50,310.50,310.50,0.883731,0.467996,576.0
2,2023-01-09,576.0,191,399.00,399.00,399.00,0.937328,-0.348448,502.0
3,2023-01-16,502.0,170,424.75,424.75,424.75,0.363429,-0.931622,558.0
4,2023-01-23,558.0,187,548.50,451.40,451.40,-0.453769,-0.891119,545.0


Concluding the cleaning and feature engineering, let's save the results into two CSV files, one for each final DataFrame.

In [121]:
df_items = df_weekly_item_quantity_encoded.copy()
df_total = df_weekly_total.copy()

processed_path = Path("data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

df_total.to_csv(processed_path / "weekly_total_clean.csv", index=False)
df_items.to_csv(processed_path / "weekly_items_clean.csv", index=False)